# Lesson 13 — Exercises

## Exercise 1 — Verify a Chvátal–Gomory cut (15 pts)

For the LP $\max x_1 + x_2$ s.t. $2 x_1 + x_2 \le 7, x_1 + 3 x_2 \le 9, x \in \mathbb{Z}^2_+$, the LP relaxation has a fractional optimum near $(2.4, 2.2)$. The provided cut

$$x_1 + x_2 \le 4$$

is a Chvátal–Gomory cut for this system (derive it by adding the two original constraints with multipliers $u = (2/5, 1/5)$ to get $x_1 + x_2 \le 23/5$, then round the RHS down to an integer. Note that $u = (1/3, 1/3)$ does **not** work: it gives $x_1 + \tfrac43 x_2 \le \tfrac{16}{3}$, which rounds to the weaker $x_1 + x_2 \le 5$). Your job is to **verify**: (a) the cut is satisfied by every integer-feasible point in $[0, 5]^2$ that satisfies the original constraints; (b) the cut is violated by the LP optimum (LHS $\approx 4.6 > 4$); (c) re-solve the LP with this extra constraint and report the new objective.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# TODO


## Exercise 2 — Cut a hole in the LP (15 pts)

For the same problem, plot the LP relaxation polytope and the integer points. Plot your Gomory cut on top. Visually confirm the cut separates $x_{LP}$ from the integer hull.

In [ ]:
# TODO: matplotlib


## Exercise 3 — MIR (15 pts)

Apply MIR rounding to one constraint of the multi-knapsack problem of your choice (say 5 items, 2 knapsacks). Add the MIR cut and re-solve. Report (i) how much the LP-relaxation bound moves (solve with continuous variables), and (ii) the node count with and without the cut on `discopt`'s own MILP tree with its root cuts switched off, `m.solve(milp_backend="native", milp_cuts=False)`, so that its own cut loop does not re-derive or mask yours. The default route hands a MILP to HiGHS, which usually reports 1 node on an instance this small.

In [ ]:
# TODO


## Exercise 4 — How much does the root cut loop buy? (10 pts)

`discopt`'s own MILP tree (`milp_backend="native"`) runs a root cut loop of cover, clique, Gomory and MIR cuts before it branches, and `milp_cuts=False` switches the loop off:

```python
r_cuts   = build().solve(milp_backend="native")                   # root cut loop on
r_nocuts = build().solve(milp_backend="native", milp_cuts=False)  # plain LP relaxation
```

Take at least three random 5-row multi-dimensional knapsacks with 40 binaries (different seeds). For both arms report nodes, wall time, the number of cuts added (`r.solver_stats` keys `"cuts/cover_clique"`, `"cuts/gomory"`, `"cuts/mir"`) and `r.root_bound`, which on this route is the bound after the cut loop. Also report the LP-relaxation gap $(z_{LP} - z^\star)/z^\star$ and the fraction of it the cuts close at the root, $(z_{LP} - \text{root bound})/(z_{LP} - z^\star)$. Do the cuts pay for themselves on every seed? Explain your result in terms of what the cuts can and cannot see on these instances, and why a tighter root bound does not always mean a smaller tree.

In [ ]:
# TODO


## Exercise 5 — Custom cut callback (15 pts)

Write a `discopt` cut callback that, at each node, generates a *cover cut* for any violated knapsack constraint. (A cover is a set $S$ with $\sum_{i \in S} w_i > W$; the cut is $\sum_{i \in S} x_i \le |S| - 1$.) Apply it to a knapsack with 15–20 items and compare node counts with and without the callback.

API (`discopt.callbacks`): the callback has the signature `cb(ctx, model)`. `ctx.x_relaxation` is the node's relaxation point (flat; the first `n` entries are `x` if `x` is the first variable you declared). It returns a list of `CutResult(terms=[(x[i], 1.0), ...], sense="<=", rhs=len(S) - 1)`, or `[]`. Pass it as `m.solve(cut_callback=cb)`.

A pure MILP with a `cut_callback` leaves the HiGHS route and runs on `discopt`'s spatial branch-and-bound, the engine that calls the callback (check `r.algorithm_route`). That engine differs from both HiGHS and the native MILP tree, so for a like-for-like baseline run it with a callback that returns no cuts: `m.solve(cut_callback=lambda ctx, model: [])`. Avoid values = weights with even weights and an odd capacity here: every objective value is then even while the LP bound sits at the odd capacity, the spatial engine does not use the fact that the objective is integral, and it runs until the time limit with an absolute gap of 1.

In [ ]:
# TODO
